# MSB observables in the persistent-fluctuation phase — dense power-law graph, competitive $\mu$

Original-model GLV $\dot x_i = x_i(1 - x_i + (Ax)_i)$ on a **dense power-law graph** ($\langle k\rangle
\approx 11$) at **competitive $\mu=-2$**. Holding $\mu$ fixed and sweeping $\sigma$, the system passes
unique-fixed-point coexistence $\to$ a **persistent-fluctuation phase** $\to$ unbounded growth. In the
fluctuating phase we measure the Moran--Secchi--Bouchaud size--volatility relation and the growth-rate
distribution.

**Locating the fluctuating phase by STATIONARITY.** A single late-window variance cannot tell a genuine
persistent fluctuation from a slow relaxation onto a static fixed point. We therefore compare
$\mathrm{Var}(\ln x)$ across two successive late windows: **flat across windows = persistent fluctuations;
decaying = relaxation.** (Verified: at $N{=}400,\langle k\rangle{\approx}11,\sigma{=}1.8$ the variance is
flat $\approx2.9\times10^{-3}$ over $t=1000\to3000$; at a relaxing point it decays many orders.)

**Measurement (post-transient).** Because the one-sided initial relaxation transient swamps the
comparatively small steady fluctuations, the growth law is measured on the **post-transient window**
(second half of the trajectory) -- justified precisely because the steady state genuinely fluctuates
there. Relative size $S_i=N x_i/\sum_j x_j$ (cross-section); $g_i=\Delta\ln S_i$;
$\sigma_i=\sqrt{\pi/2}\,\overline{|g-\bar g|}$. LSODA with a divergence-terminating event; diverged runs
dropped and counted.

In [ ]:
import multiprocessing as mp
mp.set_start_method("fork", force=True)

import os
import numpy as np
import networkx as nx
import scipy.sparse as sp
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.integrate import solve_ivp
from scipy.stats import skew, kurtosis

import glv
glv.apply_style()

N          = 400          # species / firms
ALPHA      = 1.5          # power-law degree tail: P(k) ~ k^-(1+alpha)
KMIN       = 5.0          # degree lower cutoff (sets density; <k> ~ 11)
KMAX       = 120          # structural degree cutoff
MU         = -2.0         # FIXED competitive mean interaction
SIGMA_GRID = np.round(np.array([1.0, 1.2, 1.4, 1.6, 1.7, 1.8, 1.9, 2.0]), 3)
tmax       = 3000.0       # long horizon: need persistent-fluctuation statistics
n_samples  = 600          # trajectory samples over [0, tmax] (dt = 5)
BURN       = 0.5          # discard first BURN fraction (transient) for measurement
n_ic_sweep = 3            # ICs per sigma in the phase sweep
n_runs     = 10           # graph realizations pooled at sigma*
FLOOR      = 1e-3
FLUCT      = 1e-6         # variance floor: above => fluctuating (if also stationary)
STAT_RATIO = 0.3          # Var(late2)/Var(late1) >= this => stationary (not decaying)
BLOW       = 1e6
n_workers  = min(8, (mp.cpu_count() or 2))
SWEEP_RES  = "fluct_powerlaw_sweep.npz"
MEAS_RES   = "fluct_powerlaw_meas.npz"
print(f"power-law alpha={ALPHA}, kmin={KMIN} | mu={MU} | sigma {SIGMA_GRID[0]}..{SIGMA_GRID[-1]}"
      f" | N={N} | tmax={tmax:g} | workers={n_workers}")

In [ ]:
# Inline graph + interaction builders (project convention).
def powerlaw_degrees(seed):
    rng = np.random.default_rng(seed)
    k = KMIN * rng.uniform(0.0, 1.0, N) ** (-1.0 / ALPHA)
    k = np.clip(np.round(k).astype(int), 1, KMAX)
    if k.sum() % 2:
        k[0] += 1
    return k

def build_adjacency(seed):
    k = powerlaw_degrees(seed)
    G = nx.Graph(nx.configuration_model(list(k), seed=int(seed)))
    G.remove_edges_from(nx.selfloop_edges(G))
    A = nx.to_scipy_sparse_array(G, format="csr", dtype=float).tocoo()
    return A, float(k.mean())

def make_W(A, z, C, sigma):
    return sp.csr_array((MU / C + (sigma / np.sqrt(C)) * z, (A.row, A.col)), shape=A.shape)

def integrate(W, x0):
    def rhs(t, x):
        x = np.clip(x, 0.0, None)
        return x * (1.0 - x + W @ x)
    def blow(t, x):
        return BLOW - np.max(np.clip(x, 0.0, None))
    blow.terminal = True
    blow.direction = -1
    t_eval = np.linspace(0.0, tmax, n_samples)
    sol = solve_ivp(rhs, (0.0, tmax), x0, method="LSODA", rtol=1e-7, atol=1e-9,
                    t_eval=t_eval, events=blow, max_step=3.0)
    return sol.t, np.clip(sol.y, 1e-12, None), (sol.t_events[0].size > 0)

def window_var(t, X, lo, hi):
    m = (t >= lo * tmax) & (t < hi * tmax)
    alive = X[:, m].mean(1) > FLOOR
    return float(np.median(np.log(X[alive][:, m]).var(1))) if alive.any() else np.nan

In [ ]:
# Phase sweep: classify each sigma by STATIONARITY of late-window Var(ln x).
#   v1 = Var on [0.5,0.75] tmax, v2 = Var on [0.75,1.0] tmax.
#   fluctuating  <=> bounded AND v2 > FLUCT AND v2/v1 >= STAT_RATIO (flat, not decaying).
def _sweep_job(args):
    seed, sigma = args
    A, C = build_adjacency(seed)
    z = np.random.default_rng(seed + 1).standard_normal(A.row.size)
    W = make_W(A, z, C, sigma)
    x0 = np.random.default_rng(seed + 2).uniform(0.1, 1.0, N)
    t, X, diverged = integrate(W, x0)
    if diverged:
        return sigma, np.nan, np.nan, True
    v1 = window_var(t, X, 0.5, 0.75)
    v2 = window_var(t, X, 0.75, 1.0)
    return sigma, v1, v2, False

if os.path.exists(SWEEP_RES):
    d = np.load(SWEEP_RES)
    sweep_sigma, v1_med, v2_med, div_frac = d["sweep_sigma"], d["v1_med"], d["v2_med"], d["div_frac"]
    print(f"loaded cache {SWEEP_RES}")
else:
    jobs = [(2000 + 10 * i + r, s) for i, s in enumerate(SIGMA_GRID) for r in range(n_ic_sweep)]
    v1d = {float(s): [] for s in SIGMA_GRID}
    v2d = {float(s): [] for s in SIGMA_GRID}
    divd = {float(s): 0 for s in SIGMA_GRID}
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = [ex.submit(_sweep_job, j) for j in jobs]
        for done, fut in enumerate(as_completed(futs), 1):
            sigma, v1, v2, dv = fut.result()
            if dv:
                divd[float(sigma)] += 1
            else:
                if v1 == v1: v1d[float(sigma)].append(v1)
                if v2 == v2: v2d[float(sigma)].append(v2)
            print(f"  [{done}/{len(jobs)}] sigma={sigma}: {'diverged' if dv else f'v2={v2:.2e}'}")
    sweep_sigma = np.array(SIGMA_GRID, float)
    v1_med = np.array([np.median(v1d[float(s)]) if v1d[float(s)] else np.nan for s in SIGMA_GRID])
    v2_med = np.array([np.median(v2d[float(s)]) if v2d[float(s)] else np.nan for s in SIGMA_GRID])
    div_frac = np.array([divd[float(s)] / n_ic_sweep for s in SIGMA_GRID])
    np.savez(SWEEP_RES, sweep_sigma=sweep_sigma, v1_med=v1_med, v2_med=v2_med, div_frac=div_frac)
    print(f"saved {SWEEP_RES}")

ratio = v2_med / v1_med
fluctuating = (div_frac < 0.5) & (v2_med > FLUCT) & (ratio >= STAT_RATIO)
print(f"\n{'sigma':>6} {'div':>5} {'Var(late1)':>11} {'Var(late2)':>11} {'ratio':>7}  phase")
for i, s in enumerate(sweep_sigma):
    tag = ("unbounded" if div_frac[i] >= 0.5 else
           "FLUCTUATING(stationary)" if fluctuating[i] else
           "relaxing/static" if (v2_med[i] == v2_med[i] and ratio[i] < STAT_RATIO and v1_med[i] > FLUCT)
           else "unique-FP/static")
    print(f"{s:6.2f} {div_frac[i]:4.0%} {v1_med[i]:11.2e} {v2_med[i]:11.2e} {ratio[i]:7.2f}  {tag}")

cand = np.where(fluctuating)[0]
sigma_star = float(sweep_sigma[cand[np.argmax(v2_med[cand])]]) if cand.size else float("nan")
print(f"\nsigma* (stationary fluctuating, bounded) = {sigma_star}")

In [ ]:
# Measure MSB observables at sigma* on the POST-TRANSIENT window (t >= BURN*tmax), pooled.
def run_one(seed, sigma):
    A, C = build_adjacency(seed)
    z = np.random.default_rng(seed + 1).standard_normal(A.row.size)
    W = make_W(A, z, C, sigma)
    x0 = np.random.default_rng(seed + 2).uniform(0.1, 1.0, N)
    t, X, diverged = integrate(W, x0)
    if diverged:
        return None
    m = t >= BURN * tmax
    Xl = X[:, m]
    S = N * Xl / Xl.sum(axis=0, keepdims=True)        # cross-section relative size
    log_S = np.log(np.maximum(S, 1e-15))
    g = np.diff(log_S, axis=1)
    g_bar = g.mean(axis=1, keepdims=True)
    vol = np.sqrt(np.pi / 2.0) * np.mean(np.abs(g - g_bar), axis=1)
    return S.mean(axis=1), vol, g.ravel()

def _meas_job(seed):
    return run_one(seed, sigma_star)

if os.path.exists(MEAS_RES):
    d = np.load(MEAS_RES)
    avg_all, vol_all, g_all, dropped = d["avg_all"], d["vol_all"], d["g_all"], int(d["dropped"])
    print(f"loaded cache {MEAS_RES}")
else:
    if not np.isfinite(sigma_star):
        raise RuntimeError("no stationary fluctuating sigma found; adjust SIGMA_GRID / density")
    avg_list, vol_list, g_list, dropped = [], [], [], 0
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = [ex.submit(_meas_job, 6000 + 7 * r) for r in range(n_runs)]
        for done, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            if res is None:
                dropped += 1
                print(f"  [{done}/{n_runs}] dropped (diverged)")
            else:
                a, v, g = res
                avg_list.append(a); vol_list.append(v); g_list.append(g)
                print(f"  [{done}/{n_runs}] ok ({a.size} firms)")
    avg_all = np.concatenate(avg_list); vol_all = np.concatenate(vol_list); g_all = np.concatenate(g_list)
    np.savez(MEAS_RES, avg_all=avg_all, vol_all=vol_all, g_all=g_all, dropped=dropped)
    print(f"saved {MEAS_RES}")
print(f"\nat sigma*={sigma_star} (post-transient): {avg_all.size} firms, {dropped}/{n_runs} dropped, "
      f"{g_all.size} growth samples")

In [ ]:
# Phase diagram: stationarity of late-window variance vs sigma.
fig, ax = plt.subplots(figsize=(7.8, 5.0))
ax.semilogy(sweep_sigma, np.where(v1_med > 0, v1_med, np.nan), "o-", color="#457b9d",
            label=r"Var$(\ln x)$ on $[0.5,0.75]\,t_{\max}$")
ax.semilogy(sweep_sigma, np.where(v2_med > 0, v2_med, np.nan), "s-", color="#1d3557",
            label=r"Var$(\ln x)$ on $[0.75,1.0]\,t_{\max}$")
ax.axhline(FLUCT, color="#888", ls=":", lw=1)
for s, df in zip(sweep_sigma, div_frac):
    if df >= 0.5:
        ax.axvline(s, color="#c1121f", lw=6, alpha=0.12)
if np.isfinite(sigma_star):
    ax.axvline(sigma_star, color="#2a9d8f", lw=1.6, ls="--", label=rf"$\sigma^*={sigma_star:g}$")
ax.set(xlabel=r"disorder $\sigma$", ylabel=r"late-window $\mathrm{Var}(\ln x)$",
       title=rf"Stationarity sweep at $\mu={MU}$ (power-law $\langle k\rangle\!\approx\!11$)"
             + "\n(two curves overlapping = stationary fluctuations; split/dropping = relaxing; red = unbounded)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("fluct_powerlaw_phase.png", dpi=120)
plt.show()

In [ ]:
# Plot 1: size-volatility relation at sigma* (post-transient).
def vbin(x, y, nb=22):
    o = np.argsort(x); xp, yp = x[o], y[o]
    parts = np.array_split(np.arange(xp.size), nb)
    return (np.array([xp[p].mean() for p in parts]), np.array([np.median(yp[p]) for p in parts]))

def two_slopes(x, y):
    bx, by = vbin(x, y)
    j = int(np.argmin(by))
    if j < 1 or j > len(bx) - 2:
        s = np.polyfit(np.log10(bx), np.log10(by), 1)[0]
        return bx[j], s, s
    lo = np.polyfit(np.log10(bx[:j + 1]), np.log10(by[:j + 1]), 1)[0]
    hi = np.polyfit(np.log10(bx[j:]), np.log10(by[j:]), 1)[0]
    return bx[j], lo, hi

live = (avg_all > FLOOR) & (vol_all > 0) & np.isfinite(vol_all)
bx, by = vbin(avg_all[live], vol_all[live])
smin, blo, bhi = two_slopes(avg_all[live], vol_all[live])
fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.loglog(avg_all[live], vol_all[live], ".", ms=2, color="#b8c6d6", alpha=0.3)
ax.loglog(bx, by, "o-", ms=5, color="#1d3557", label="binned median")
ax.set(xlabel=r"time-average relative size $\bar S_i$",
       ylabel=r"growth-rate volatility $\sigma_i=\sqrt{\pi/2}\,\overline{|g-\bar g|}$",
       title=rf"Size--volatility, fluctuating phase ($\mu={MU}$, $\sigma^*={sigma_star:g}$, post-transient)"
             + "\n" + rf"slopes: descending $\beta_{{\rm lo}}={blo:+.2f}$, ascending $\beta_{{\rm hi}}={bhi:+.2f}$")
ax.grid(True, which="both", alpha=0.15)
ax.legend()
plt.tight_layout()
plt.savefig("fluct_powerlaw_volatility_size.png", dpi=120)
plt.show()
print(f"V-minimum at S~{smin:.3g}; descending beta={blo:+.3f}, ascending beta={bhi:+.3f}")

In [ ]:
# Plot 2: growth-rate distribution (expect TWO-TAILED) vs Gaussian and Laplace.
g = g_all[np.isfinite(g_all)]
mu_g, sd_g, med_g = g.mean(), g.std(), np.median(g)
b_lap = np.mean(np.abs(g - med_g))
gc = g - med_g
r_tail, l_tail = np.mean(gc > 2 * sd_g), np.mean(gc < -2 * sd_g)
lo, hi = np.percentile(g, [0.2, 99.8])
edges = np.linspace(lo, hi, 80); centers = 0.5 * (edges[:-1] + edges[1:])
dens, _ = np.histogram(g, bins=edges, density=True)
gauss = np.exp(-0.5 * ((centers - mu_g) / sd_g) ** 2) / (sd_g * np.sqrt(2 * np.pi))
lap = np.exp(-np.abs(centers - med_g) / b_lap) / (2 * b_lap)
fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.semilogy(centers, np.where(dens > 0, dens, np.nan), "o", ms=3, color="#1d3557",
            label=r"pooled $g=\Delta\ln S$")
ax.semilogy(centers, gauss, "-", color="#457b9d", lw=1.5, label=r"Gaussian (same $\sigma$)")
ax.semilogy(centers, lap, "--", color="#c1121f", lw=1.5, label="Laplace (same scale)")
ax.axvline(med_g, color="#888", lw=1, ls=":")
ax.set(xlabel=r"growth rate $g=\Delta\ln S_i$", ylabel="density",
       title=rf"Growth-rate distribution, fluctuating phase ($\mu={MU}$, $\sigma^*={sigma_star:g}$)")
ax.grid(True, which="both", alpha=0.15)
ax.legend()
plt.tight_layout()
plt.savefig("fluct_powerlaw_growth_dist.png", dpi=120)
plt.show()
print(f"growth: mean={mu_g:+.4f}, median={med_g:+.4f}, std={sd_g:.4f}, skew={skew(g):+.2f}, "
      f"excess kurtosis={kurtosis(g):.2f}")
print(f"  tail mass beyond 2 std: right(growth)={100*r_tail:.2f}%  left(decay)={100*l_tail:.2f}%")

## Summary

_Filled in after execution._